# 天気図パターン分類 - 推論専用

学習済みモデル(Google Drive上の`model.pt`)を使って、アップロードした天気図画像が
どの気圧配置パターンに近いかを推論します。学習は行いません。

上から順にセルを実行してください。

## 1. セットアップ

In [ ]:
REPO_URL = "https://github.com/awg-yk/weather-pattern-classification.git"
BRANCH = "claude/weather-chart-classification-4b6in1"
REPO_DIR = "/content/weather-pattern-classification"

# 学習済みモデルの重みのパス。ご自身のDrive上の実際の保存先に合わせて変更してください。
WEIGHTS_PATH = "/content/drive/MyDrive/weather-pattern-classification-data/weights/model.pt"

import subprocess, os

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(WEIGHTS_PATH), f"モデルの重みが見つかりません: {WEIGHTS_PATH}"
print("セットアップ完了。モデル:", WEIGHTS_PATH)

## 2. モデルを読み込む(このセッションで1回だけでOK)

In [ ]:
import sys
sys.path.append(REPO_DIR)

import torch
from src.labels import INDEX_TO_LABEL, LABEL_JA, LABELS
from src.model import build_model
from src.train import get_transforms
from scripts.preprocess_jma import DEFAULT_STAMP_BOX, autocrop_to_content, mask_stamp_box

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(num_classes=len(LABELS))
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.to(device)
model.eval()

transform = get_transforms(train=False)
print("モデル読み込み完了")

## 3. 画像をアップロードして推論

このセルを実行するたびに新しい画像をアップロードして分類できます。

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

THRESHOLD = 0.5  # この確信度を超えたラベルを「該当する」として表示する

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

raw_image = Image.open(image_path).convert("RGB")
# 気象庁の生のPDF変換画像(枠・座標グリッド・日時スタンプ付き)を想定した前処理。
# 既に前処理済みの画像を使う場合はこの2行をコメントアウトしてください。
display_image = autocrop_to_content(raw_image)
display_image = mask_stamp_box(display_image, DEFAULT_STAMP_BOX)

input_tensor = transform(display_image).unsqueeze(0).to(device)
with torch.no_grad():
    probs = torch.sigmoid(model(input_tensor))[0].cpu()

sorted_indices = torch.argsort(probs, descending=True)
predicted = [INDEX_TO_LABEL[i.item()] for i in sorted_indices if probs[i] > THRESHOLD]

plt.figure(figsize=(5, 5))
plt.imshow(display_image)
plt.axis("off")
title = " / ".join(LABEL_JA[l] for l in predicted) if predicted else "該当なし"
plt.title(f"予測: {title}")
plt.show()

print("--- 全ラベルの確信度 ---")
for i in sorted_indices:
    label = INDEX_TO_LABEL[i.item()]
    print(f"{LABEL_JA[label]}: {probs[i].item() * 100:.1f}%")